# Lakehouse Erasure Lab — Results

This notebook analyzes reproducible deletion benchmarks from the local
DuckDB/Parquet implementation.

The experiment measures current-state deletion latency across four
synthetic dataset sizes. Each size is tested three times.

Historical snapshot expiry is a separate operation and is not included
in deletion latency measurements.

These results do not represent Apache Iceberg or production vector
database performance.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "results" / "scale_benchmark.csv").exists():
    ROOT = Path.cwd().parent

csv_path = ROOT / "results" / "scale_benchmark.csv"
df = pd.read_csv(csv_path)
display(df)

In [ ]:
print("Dataset sizes:", len(df))
print("Total runs:", int(df["repeats"].sum()))
print("Verified runs:", int(df["verified_runs"].sum()))
print("All runs verified:", bool(df["all_runs_verified"].all()))

assert df["all_runs_verified"].all()
assert (df["mean_latency_ms"] > 0).all()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(df["users"], df["mean_latency_ms"], marker="o")
plt.xlabel("Number of synthetic users")
plt.ylabel("Mean deletion latency (ms)")
plt.title("Deletion Latency vs Dataset Size")
plt.grid(True, alpha=0.3)
plt.tight_layout()

chart_path = ROOT / "results" / "deletion_latency.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Chart saved to {chart_path}")

In [ ]:
summary = df[
    [
        "users",
        "repeats",
        "mean_latency_ms",
        "median_latency_ms",
        "verified_runs",
        "all_runs_verified",
    ]
]

summary.to_csv(ROOT / "results" / "benchmark_summary.csv", index=False)
display(summary)